# 13.1 偏好為什麼必須連同問題一起記？


對「只回數字：2+2=?」，A回答「4」，B回答「4，就像兩雙筷子共有四根」。兩份都算對，只有A符合只回數字。若要求改成生活比喻，B又可能較合適。**偏好**是在同一個任務條件下，標記哪份回答更合適，不是把一種寫法永久封為好答案。

先寫本次判準：「先正確，再遵循只回數字」，再保留提問、兩份原文、選擇和原因。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
pair = {"prompt": "只回數字：2+2=?", "A": "4", "B": "4，就像兩雙筷子共有四根。"}
rule = "先正確，再遵循只回數字"
chosen = "A"
print("條件", pair["prompt"])
print("比較", pair["A"], "／", pair["B"])
print("理由", rule, "選擇", chosen)

條件 只回數字：2+2=?
比較 4 ／ 4，就像兩雙筷子共有四根。
理由 先正確，再遵循只回數字 選擇 A


`chosen`直接寫A，是我們的人工判斷；程式只印出紀錄，不會讀文排序。保存原因讓後來的人知道，A勝出是因限制，不是所有比喻都不好。

兩份都不合適時，較佳的一份也不等於理想示範；沒有清楚差別時，則可以記平手或不確定。否則不同標註者各猜一套標準，訓練訊號就混在一起。

練習只刪掉提問裡的「只回數字」。先重寫偏好理由，再修改規則和標記。原程式的A不會自動改：問題變了，需要重新判斷。後面的數學先用最清楚的加法正誤對，例如同題4對5；格式偏好則另用兩份都算對的答案比較。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

我們真正訓練時先收窄問題，只比較加法真值。例如同一題`2+2=?`，較佳候選是`4`，較差候選是`5`，不是讓模型自訂什麼叫好。兩個加數都從0到7挑選，因此有8×8＝64道題，每題配一組較佳／較差回答。切分時，把交換加數後相同的題目算成一個「家族」：例如`2+3=?`與`3+2=?`必須留在同一組，不能一題拿來教、另一題拿來考，否則測試可能只是在考已見過的數字組合。整個家族一起分配後，本輪共有49對拿來訓練、8對留作驗證，另外7對作最後檢查。訓練組用來更新權重；驗證組用來觀察訓練途中是否學得更好；最後檢查組用來檢查未教過的組合，不能拿它的分數來選訓練設定。本次程式預先寫定兩種訓練設定與更新次數，也在訓練前記錄起點的檢查分數，並在每一支訓練後再測一次；檢查資料不參與權重更新，程式也不依這些分數改設定。這與只在全部訓練結束後開考卷的流程不同，因此閱讀結果時，要分清「拿檢查題量前後變化」與「看了檢查分數再調參數」。這三組與[8.3的加法起點](https://birdhackor.github.io/tiny-perceptron-vlm/8.3.html)使用相同問題切分，讓前後比較對得起來。另做一條格式支線，把兩篇都算對的回答拿來比較，見[13.7](https://birdhackor.github.io/tiny-perceptron-vlm/13.7.html)。先分清這兩種理由，後面才知道模型是改了內容排序，還是只偏好沒有附加句的寫法。

</details>
